### Final Project

### CCDB June-2026 Report

### Set Up

In [1]:
# --- Setup ---------------------------------------------------------------
# Core libraries only; model/API imports get appended once we pick the GenAI tool.
import hashlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

# Notebook lives in notebooks/; resolve paths from the project root so it runs on either partner's machine
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_DIR / "data" / "raw" / "CCDB_Export_19_June_2026.zip"

# Reference fingerprint of the June 2026 export, recorded when we first profiled it (2026-10-24).
# The load step recomputes the hash from the file and compares it to this value.
EXPECTED_SHA256 = "3021c8af9eeed4971a92f15be21c7ce08cf3c1337151de255e5b12bf782049df"

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__, "| numpy:", np.__version__)
assert DATA_PATH.exists(), f"Data file not found: {DATA_PATH}"

Python: 3.12.10
pandas: 3.0.5 | numpy: 2.5.3


### Load Data

In [2]:
# --- Load data -----------------------------------------------------------
# Integrity check: recompute the file's fingerprint and compare it to the recorded one
actual_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
sha_match = actual_sha256 == EXPECTED_SHA256

print(f"File:            {DATA_PATH.name}")
print(f"Expected sha256: {EXPECTED_SHA256}")
print(f"Actual sha256:   {actual_sha256}")
print(f"Integrity check: {'PASS' if sha_match else 'FAIL'}")
assert sha_match, "Data file differs from the profiled export; do not continue"

# Read every column as text; types get converted on purpose in the cleaning step
df_raw = pd.read_csv(DATA_PATH, dtype=str, low_memory=False)

EXPECTED_COLS = [
    "Date received", "Product", "Sub-product", "Issue", "Sub-issue",
    "Consumer complaint narrative", "Company public response", "Company",
    "State", "ZIP code", "Tags", "Submitted via", "Date sent to company",
    "Company response to consumer", "Timely response?", "Complaint ID",
]
assert list(df_raw.columns) == EXPECTED_COLS, "Column layout differs from the profiled export"
assert len(df_raw) == 620_794, f"Unexpected row count: {len(df_raw):,}"
assert df_raw["Complaint ID"].is_unique, "Duplicate Complaint IDs"

print(f"\nLoaded {len(df_raw):,} rows x {df_raw.shape[1]} columns "
      f"({df_raw.memory_usage(deep=True).sum() / 1e6:,.0f} MB in memory)")

# Preview without the narrative column to keep the output short
df_raw.drop(columns=["Consumer complaint narrative"]).head(3)

File:            CCDB_Export_19_June_2026.zip
Expected sha256: 3021c8af9eeed4971a92f15be21c7ce08cf3c1337151de255e5b12bf782049df
Actual sha256:   3021c8af9eeed4971a92f15be21c7ce08cf3c1337151de255e5b12bf782049df
Integrity check: PASS

Loaded 620,794 rows x 16 columns (670 MB in memory)


,Date received,Product,Sub-product,Issue,Sub-issue,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2026-06-07,Credit card,General-purpose credit card or charge card,Closing your account,Company closed your account,NaN,OneMain Finance Corporation,TN,37087,NaN,Web,2026-06-07,Closed with explanation,Yes,22972138
1,2026-06-06,Credit reporting or other personal consumer reports,Credit reporting,Improper use of your report,Credit inquiries on your report that you don't recognize,NaN,"EQUIFAX, INC.",MI,48221,NaN,Web,2026-06-06,Closed with explanation,Yes,22960234
2,2026-06-02,Debt collection,Other debt,Electronic communications,"You told them to stop contacting you, but they keep trying",NaN,"Landmark Strategy Group, LLC",NC,28278,NaN,Web,2026-08-28,Untimely response,No,22819786
